In [1]:
# Importy 

import tensorflow as tf
from keras import layers, models
import matplotlib.pyplot as plt
import numpy as np 
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, log_loss
from sklearn.preprocessing import label_binarize


from keras.applications.vgg16 import VGG16, preprocess_input as vgg16_prep
from keras.applications.vgg19 import VGG19, preprocess_input as vgg19_prep
from keras.applications.resnet50 import ResNet50, preprocess_input as resnet_prep
from keras.applications.inception_v3 import InceptionV3, preprocess_input as incep_prep
from keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input as mobile_prep
from keras.applications.efficientnet import EfficientNetB0, preprocess_input as eff_prep

In [2]:
# Wczytanie danych

# Ścieżki
train_dir = r"C:\Users\mkrol\PycharmProjects\GGSN_labolatory\project02\data\seg_train\seg_train"
pred_dir = r"C:\Users\mkrol\PycharmProjects\GGSN_labolatory\project02\data\seg_pred\seg_pred"
test_dir = r"C:\Users\mkrol\PycharmProjects\GGSN_labolatory\project02\data\seg_test\seg_test"

# Wczytanie danych z opisanymi podfolderami
train_ds = tf.keras.utils.image_dataset_from_directory(
    directory=train_dir,        # ścieżka
    image_size=(150, 150),      # wymiary obrazu
    batch_size = 32,            # liczba obrazów przetwarzana w jednym momencie
    label_mode='int'            # lub 'categorical' dla One-Hot Encoding
)

# Wczytanie danych z opisanymi podfolderami
val_ds = tf.keras.utils.image_dataset_from_directory(
    directory=test_dir,          # ścieżka
    image_size=(150, 150),      # wymiary obrazu
    batch_size = 32,            # liczba obrazów przetwarzana w jednym momencie
    label_mode='int'            # lub 'categorical' dla One-Hot Encoding
)

# Wczytanie danych bez podfolderu
pred_ds = tf.keras.utils.image_dataset_from_directory(
    directory=pred_dir,         # ścieżka
    labels=None,                # informuje że nie ma podfolderów
    label_mode=None,            # informuje że nie ma podfolderów
    image_size=(150, 150),      # wymiary obrazu
    batch_size = 32,            # liczba obrazów przetwarzana w jednym momencie
    shuffle=False               # konieczne przy wczytaniu danych do tasowania, bo algorytm musi wiedziec ktore zdjecie jest do ktorej etykiety
)

Found 14034 files belonging to 6 classes.
Found 3000 files belonging to 6 classes.
Found 7301 files.


In [3]:
# Augmentacja danych -> będziemy ją wykonywać w locie. Polega to na tym, że model w każej epocie widzi to samo zdjęcie, ale za każdym razem przekształcone

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),        # Lustrzane odbicie lewo-prawo
    layers.RandomRotation(0.2),             # Obrót o max 20%
    layers.RandomZoom(0.2),                 # Przybliżenie/oddalanie o 20%
    layers.RandomContrast(0.2),             # Zmiana kontrastu
    layers.RandomTranslation(0.1, 0.1),     # Przesunięcie o max 10% szerokości/wysokości
    layers.RandomBrightness(0.2)            # Losowo zmienia jasność o 20%
])

In [4]:
# Tworzymy model CNN
# Liczbę filtrów zwiększamy (Conv2D)
model = models.Sequential([
    # Wejście 
    layers.Input(shape=(150, 150, 3)),  # Rozmiar obrazów
    data_augmentation,                  # Augmentacja obrazów
    layers.Rescaling(1./255),           # Skalowanie do zakresu od 0 do 1
    # Warstwa 1
    layers.Conv2D(64, (3, 3), activation='relu'),   # tworzy 64 wersje obrazu, (3, 3) - wyłapywanie cech
    layers.MaxPooling2D(2, 2),                      # zmniejsza do rozmiaru 75x75
    # Warstwa 2
    layers.Conv2D(128, (3, 3), activation='relu'),  # tworzy 128 wersje obrazu, (3, 3) - wyłapywanie cech
    layers.MaxPooling2D(2, 2),                      # zmniejsza do rozmiaru 37x37
    # Warstwa 3
    layers.Conv2D(256, (3, 3), activation='relu'),  # tworzy 256 wersje obrazu, (3, 3) - wyłapywanie cech
    layers.MaxPooling2D(2, 2),                      # zmniejsza do rozmiaru 18x18
    # Warstwa 4
    layers.Conv2D(512, (3, 3), activation='relu'),  # tworzy 512 wersje obrazu, (3, 3) - wyłapywanie cech
    layers.MaxPooling2D(2, 2),                      # zmniejsza do rozmiaru 9x9
    # Warstwa klasyfikacyjna
    layers.Flatten(),                       # Zamiana map cech na wektor, bo Dense nie rozumieją 2D
    layers.Dense(512, activation='relu'),   # Głowne liczenie wyniku
    layers.Dropout(0.5),                    # Zapobiega overfittingu
    layers.Dense(6, activation='softmax')   # 6 klas wyjściowych
])

In [5]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [6]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 148, 148, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 15, 15, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    12,845,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         3,078 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,399,622 (54.93 MB)

 Trainable params: 14,399,622 (54.93 MB)

 Non-trainable params: 0 (0.00 B)

In [1]:
history = model.fit(
    train_ds,                   # Zbiór treningowy
    steps_per_epoch=150,        # Ilość kroków na epokę
    epochs=50,                  # Ilość epok
    validation_data=val_ds,     # Zbiór do walidacji
    validation_steps=50         # Ilość kroków walidacji
)

y_pred_1 = model.predict(pred_ds)
y_pred_1 = np.argmax(y_pred_1, axis=1)

NameError: name 'model' is not defined

In [ ]:
# Transfer Learning - VGG 16

# Pobranie bazy VGG16
base_model_vgg16 = VGG16(
    weights='imagenet', 
    include_top=False, 
    input_shape=(150, 150, 3)
)

# Zamrożenie wag
base_model_vgg16.trainable = False 

# Zbudowanie modelu VGG16
model_vgg16 = models.Sequential([
    layers.Input(shape=(150, 150, 3)),      # Podanie rozmiaru danych
    data_augmentation,                      # Augmentacja
    layers.Lambda(vgg16_prep),        # Specjalne przygotowanie pod VGG
    base_model_vgg16,                       # Mechanizm VGG16
    layers.GlobalAveragePooling2D(),        # Zamiana map cech na wektor
    layers.Dense(256, activation='relu'),   # Głowne liczenie wyniku
    layers.Dropout(0.3),                    # Zapobiega overfittingowi
    layers.Dense(6, activation='softmax')   # Zwrócie wartości jednej z 6 klas
])

model_vgg16.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

history_vgg16 = model_vgg16.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

y_pred_2 = model_vgg16.predict(pred_ds)
y_pred_2 = np.argmax(y_pred_2, axis=1)

In [ ]:
# Transfer Learning - VGG 19

# Pobranie bazy VGG16
base_model_vgg19 = VGG19(
    weights='imagenet', 
    include_top=False, 
    input_shape=(150, 150, 3)
)

# Zamrożenie wag
base_model_vgg19.trainable = False 

# Zbudowanie modelu VGG16
model_vgg19 = models.Sequential([
    layers.Input(shape=(150, 150, 3)),      # Podanie rozmiaru danych
    data_augmentation,                      # Augmentacja
    layers.Lambda(vgg19_prep),        # Specjalne przygotowanie pod VGG
    base_model_vgg19,                       # Mechanizm VGG16
    layers.GlobalAveragePooling2D(),        # Zamiana map cech na wektor
    layers.Dense(256, activation='relu'),   # Głowne liczenie wyniku
    layers.Dropout(0.3),                    # Zapobiega overfittingowi
    layers.Dense(6, activation='softmax')   # Zwrócie wartości jednej z 6 klas
])

# Wybór zasad nauki
model_vgg19.compile(
    optimizer='adam',                           # Optymizer, pozwala uczyć się na błędach
    loss='sparse_categorical_crossentropy',     # Licznik błędów
    metrics=['accuracy']                        # Metryka
)

# Zapisanie danych o uczeniu
history_vgg19 = model_vgg19.fit(
    train_ds,                                   # Zbiór treningowy
    validation_data=val_ds,                     # Zbiór do walidacji
    epochs=15                                   # Liczba epok
)

y_pred_3 = model_vgg19.predict(pred_ds)
y_pred_3 = np.argmax(y_pred_3, axis=1)

In [ ]:
base_resnet = ResNet50(
    weights='imagenet', 
    include_top=False, 
    input_shape=(150, 150, 3)
)

base_resnet.trainable = False

model_resnet = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    data_augmentation,
    layers.Lambda(resnet_prep),
    base_resnet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(6, activation='softmax')
])

model_resnet.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

history_resnet = model_resnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

y_pred_4 = model_resnet.predict(pred_ds)
y_pred_4 = np.argmax(y_pred_4, axis=1)

In [ ]:
base_inception = InceptionV3(
    weights='imagenet', 
    include_top=False, 
    input_shape=(150, 150, 3)
)

base_inception.trainable = False

model_inception = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    data_augmentation,
    layers.Lambda(incep_prep),
    base_inception,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dense(6, activation='softmax')
])

model_inception.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

history_inception = model_inception.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

y_pred_5 = model_inception.predict(pred_ds)
y_pred_5 = np.argmax(y_pred_5, axis=1)


In [ ]:
base_mobilenet = MobileNetV2(
    weights='imagenet', 
    include_top=False, 
    input_shape=(150, 150, 3)
)

base_mobilenet.trainable = False

model_mobilenet = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    data_augmentation,
    layers.Lambda(mobile_prep),
    base_mobilenet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(6, activation='softmax')
])

model_mobilenet.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

history_mobilenet = model_mobilenet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

y_pred_6 = model_mobilenet.predict(pred_ds)
y_pred_6 = np.argmax(y_pred_6, axis=1)


In [ ]:
base_efficientnet = EfficientNetB0(
    weights='imagenet', 
    include_top=False, 
    input_shape=(150, 150, 3)
)

base_efficientnet.trainable = False

model_efficientnet = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    data_augmentation,
    layers.Lambda(eff_prep),
    base_efficientnet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(6, activation='softmax')
])

model_efficientnet.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

history_efficientnet = model_efficientnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

y_pred_7 = model_efficientnet.predict(pred_ds)
y_pred_7 = np.argmax(y_pred_7, axis=1)

In [ ]:
# Stworzenie czystego zbioru testowego do ewaluacji
val_ds_eval = tf.keras.utils.image_dataset_from_directory(
    test_dir, 
    image_size=(150, 150),
    batch_size=32,
    shuffle=False,
    label_mode='int'
)

# Pobranie prawdziwych klas i nazw kategorii
y_true = np.concatenate([y for x, y in val_ds_eval], axis=0)
class_names = train_ds.class_names

# Zebranie wszystkich modeli i obliczenie prawdopodobieństw
all_models = [model, model_vgg16, model_vgg19, model_resnet, model_inception, model_mobilenet, model_efficientnet]
model_names = ['My Model', 'VGG16', 'VGG19', 'ResNet', 'Inception', 'MobileNet', 'EfficientNet']
histories = [
    history, 
    history_vgg16, 
    history_vgg19, 
    history_resnet, 
    history_inception, 
    history_mobilenet, 
    history_efficientnet
]

print("Generowanie predykcji dla wszystkich modeli...")
y_probs_list = [m.predict(val_ds_eval) for m in all_models]

In [ ]:
def plot_compare_histories(m=model_names, h=histories):
    plt.figure(figsize=(16, 6))
    plt.subplot(1, 2, 1)
    for i in range(len(h)):
        plt.plot(h[i].history['val_accuracy'], label=f'{m[i]} ({max(h[i].history["val_accuracy"]):.2f})')
    plt.title('Dokładność Walidacyjna (Validation Accuracy)')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    for i in range(len(h)):
        plt.plot(h[i].history['val_loss'], label=f'{m[i]} ({min(h[i].history["val_loss"]):.2f})')
    plt.title('Strata Walidacyjna (Validation Loss)')
    plt.legend()
    plt.show()

plot_compare_histories()

In [ ]:
def plot_all_confusion_matrices(names, probs, y_true, classes):
    n = len(names)
    cols = 3
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(20, 5 * rows))
    
    for i in range(n):
        plt.subplot(rows, cols, i + 1)
        y_pred = np.argmax(probs[i], axis=1)
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
        plt.title(f'Model: {names[i]}')
    plt.tight_layout()
    plt.show()

plot_all_confusion_matrices(model_names, y_probs_list, y_true, class_names)

In [ ]:
def print_advanced_stats(names, probs, y_true, classes):
    print(f"\n{'MODEL':<15} | {'LOG LOSS':<10} | {'BRIER SCORE':<12} | {'TOP-2 ACC':<10}")
    print("-" * 60)
    
    y_true_bin = label_binarize(y_true, classes=range(len(classes)))
    
    for i in range(len(names)):
        # Log Loss 
        ll = log_loss(y_true, probs[i])
        
        # Brier Score
        bs = np.mean(np.sum((y_true_bin - probs[i])**2, axis=1))
        
        # Top-2 Accuracy
        top2_m = tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2)
        top2_m.update_state(y_true, probs[i])
        t2 = top2_m.result().numpy()
        
        print(f"{names[i]:<15} | {ll:<10.4f} | {bs:<12.4f} | {t2:<10.4f}")

print_advanced_stats(model_names, y_probs_list, y_true, class_names)

In [ ]:
best_idx = -1 # EfficientNet
print(f"\nSZCZEGÓŁOWY RAPORT DLA MODELU: {model_names[best_idx]}")
print(classification_report(y_true, np.argmax(y_probs_list[best_idx], axis=1), target_names=class_names))

In [ ]:
def plot_activation_maps(model, img_batch):
    conv_layers = [l.output for l in model.layers if isinstance(l, tf.keras.layers.Conv2D)]
    activation_model = tf.keras.models.Model(inputs=model.input, outputs=conv_layers)
    
    activations = activation_model.predict(img_batch[0:1])
    first_layer = activations[0]
    
    plt.figure(figsize=(16, 4))
    for i in range(8):
        plt.subplot(1, 8, i + 1)
        plt.imshow(first_layer[0, :, :, i], cmap='viridis')
        plt.axis('off')
    plt.suptitle("Mapy Aktywacji (Pierwsza warstwa Twojego modelu)")
    plt.show()

for img_b, _ in val_ds_eval.take(1):
    plot_activation_maps(model, img_b)